In [2]:
from pptx import Presentation
from pandas import read_csv
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.dml import MSO_THEME_COLOR
from datetime import datetime
import requests
import json

In [3]:
# Author: Matsuru Hoshi
# Date: 202509009

# This script will generate a PowerPoint (pptx) presentation for Energia Insurance.
# The goal is to generate a PowerPoint presentation with insurance quote information from the insurance company.
# The insurance company quote information is to be accessed with the insurance company API.

FONT = 'Segoe UI'

prs = Presentation('template.pptx')
purple = RGBColor.from_string("ac6edf")
# cover_info = prs.slides[0].shapes[1].table
# cover_info_cell = prs.slides[0].shapes[1].table.cell(0,0).text

In [42]:
df

,Submission Date,Nome completo,E-mail,Idioma de preferência para consultar as apólices canadenses🇨🇦:,Como ficou sabendo sobre nós?,Data de preenchimento do formulário,Quantas segurados ?,Nome completo.1,Data de nascimento 1,Idade,...,Qual o país de destino?,Qual a província de destino?,Qual a província de residência?,Todos os requerentes desse seguro estão cobertos pelo Plano de Saúde Público Canadense? - RAMQ (Québec) / OHIP (Ontario) / AHCIP (Alberta),Qual o seu tipo de visto?,Data de embarque no local de origem:,Data para iniciar o seguro:,Data de retorno (fim do seguro):,Dias de cobertura,Fique à vontade caso queira deixar uma informação adicional:
0,"Sep 24, 2025",Dea Test,andrea.britto@gmail.com,Inglês 🇺🇸,Outro,"Sep 24, 2025",1,Dea Test,"Nov 29, 1980",44.82,...,Canadá,Ontário,_Fora do Canadá,Não,Turista ou eTA,"Sep 25, 2025","Sep 25, 2025","Oct 4, 2025",10,NaN


In [10]:
# df from jotform
df = read_csv('client-jotform.csv')
# select relevant columns to keep from jotform
columns_to_keep = [df.columns[1], df.columns[2], df.columns[5], df.columns[34], df.columns[35], df.columns[36], df.columns[29], df.columns[30], df.columns[31], df.columns[6], df.columns[7], df.columns[8], df.columns[12], df.columns[13], df.columns[16], df.columns[17], df.columns[20], df.columns[21], df.columns[24], df.columns[25]]
client_df = df[columns_to_keep]
# renaming columns
client_df.columns = ["full_name", "email", "submit", "trip_start", "dest_arrival", "end", "dest_country", "dest_region", "depart_region", "num_travellers", "name_1", "dob_1", "name_2", "dob_2", "name_3", "dob_3", "name_4", "dob_4", "name_5", "dob_5"]
client_dict = client_df.to_dict('records')[0]

# Reformatting dates
old_format = "%b %d, %Y"
new_format = "%Y-%m-%d"

# Reformatting start and end dates
submit_string = client_dict["submit"] # get string
submit_date_obj = datetime.strptime(submit_string, old_format) #create obj
depart_string = client_dict["trip_start"] # get string
depart_date_obj = datetime.strptime(depart_string, old_format) #create obj
start_string = client_dict["dest_arrival"] # get string
start_date_obj = datetime.strptime(start_string, old_format) #create obj
end_string = client_dict["end"] # get string
end_date_obj = datetime.strptime(end_string, old_format)  #created obj
# setting new format
client_dict["submit"] = submit_date_obj.strftime(new_format)
client_dict["trip_start"] = depart_date_obj.strftime(new_format)
client_dict["dest_arrival"] = start_date_obj.strftime(new_format)
client_dict["end"] = end_date_obj.strftime(new_format)

# Reformatting dob dates
for x in range(1,client_dict["num_travellers"]+1):
    dob_string = client_dict[f"dob_{x}"]
    dob_date_obj = datetime.strptime(dob_string, old_format) #create obj
    client_dict[f"dob_{x}"] = dob_date_obj.strftime(new_format) # setting new format


In [13]:
client_dict

{'full_name': 'Dea Test',
 'email': 'andrea.britto@gmail.com',
 'submit': '2026-01-09',
 'trip_start': '2026-02-01',
 'dest_arrival': '2026-02-02',
 'end': '2026-03-08',
 'dest_country': 'Canadá',
 'dest_region': 'Ontário',
 'depart_region': '_Fora do Canadá',
 'num_travellers': 1,
 'name_1': 'Dea Test',
 'dob_1': '1980-11-29',
 'name_2': nan,
 'dob_2': nan,
 'name_3': nan,
 'dob_3': nan,
 'name_4': nan,
 'dob_4': nan,
 'name_5': nan,
 'dob_5': nan}

In [ ]:
DEV_CLIENT_ID = ''
DEV_CLIENT_SECRET = ''
UAT_CLIENT_ID = ''
UAT_CLIENT_SECRET = ''

def get_access_token(endpoint_url, client_ID, client_secret):

    # API parameters
    access_token_params = {
        'grant_type' : 'password',
        'user_name' : 'ENE000',
        'password' : 'Test1234!',
        'client_id' : client_ID,
        'client_secret' : client_secret
    }

    headers = {
        'content-type' : 'application/x-www-form-urlencoded',
        'user-agent' : 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:146.0) Gecko/20100101 Firefox/146.0'
    }

    session = requests.Session()
    token_response = session.post(endpoint_url, data=access_token_params, headers=headers)

    if token_response.status_code == 200:
        return token_response.json()['accessToken']
    else:
        raise Exception(f'{token_response.status_code} Could not get access token\n{token_response.reason}\n{token_response.content}')

In [45]:
def get_quote(endpoint_url, client_ID, coverage_level, access_token, client=None):
    """
    Returns quotes price for the following plan paramteres:
    - VISITOR
    - PR-VIS-1 (Product Line Code)
    - PL-SMED-21 (Plan Code for 'Visitors to Canada Medical' Plan)
    - Assumes 'YES' answer for QU-VEQ-4 questionnaire (need to find out what is answering)
    """

    # get all insured people on trip
    insured_persons = []
    for person_num in range(1,client["num_travellers"]+1):
        person_obj = {
            'insuredType' : 'VISITOR',
            'birthDate' : client[f"dob_{person_num}"],
            'questionnaires':[
                {
                    'code':'QU-VEQ-4',
                    'questions':[
                        {
                            'code':'QT-VIS-2', # Question for visitors (not sure what is, need to find out)
                            # answers are likely 2 options (YES/NO), a1 is YES, a2 is NO (if set -> ineligible)
                            'answers':[
                                {
                                    'code':'q1a1', # YES (just know must be set to YES / 1 to be eligible for plan)
                                    'value':1
                                }
                            ]
                        }
                    ]
                }
            ],
            'plansToPrice' : [
                {
                'planCode' : 'PL-SMED-21', # code for 'Visitors to Canada Medical' plan
                'priceInputParameters' : [
                        {
                            'code' : 'SUMM', # code to set coverage level
                            'value' : coverage_level # coverage value (e.g., 10k, 25k, 50k, 100k, 200k, 300k, 500k)
                        }
                    ]
                }
            ]
        }
        # append to list
        insured_persons.append(person_obj)

    # quotes request parameters dictionary
    # reformatting dates

    quotes_params = {
        'partnerCode' : 'ENE000',
        'productLineCode' : 'PR-VIS-1', # visitor product line (visitors to Canada)
        'insuredType' : 'VISITOR',
        'trip' : {
            'startDate' : client["trip_start"], #when trip starts (note: insurance only starts at destination)
            'endDate' : client["end"], #trip end date (i think is when come back to home)
            'bookingDate': client["submit"], #assuming request submission date is booking date of trip
            'arrivalDate': client["dest_arrival"], #expected date of arrival at destination, is when insurance starts
            'destination': client["dest_region"]
            # 'departureProvince' : ''
        },
        'insuredPersons' : insured_persons
    }

    # API request header
    quotes_headers = {
        'content-type' : 'application/json',
        'X-Auth-API-Key' : client_ID,
        'Authorization' : f'Bearer {access_token}',
    }

    # API post request call to retrieve information
    quotes_response = requests.post(endpoint_url, json=quotes_params, headers=quotes_headers)
    
    # Checking response status and returning retrieved information
    if quotes_response.status_code == 200:
        # return quotes_response.json()['availablePlanPrices'][0]['ratedPrice']['total']
        if 'messages' in quotes_response.json():
            raise Exception(f'{quotes_response.json()}')
        else:
            return quotes_response
    else:
        raise Exception(f'{quotes_response} {quotes_response.reason}\n{quotes_response.content}')


In [47]:
dev_endpoint_url = 'https://dev-api.tugo.com/v1/venture/accessToken'
uat_endpoint_url = 'https://uat-api.tugo.com/v1/venture/accessToken'

# endpoint url of travel insurance API
quotes_endpoint_url = 'https://uat-api.tugo.com/v1/venture/quotes/price'

access_token = get_access_token(uat_endpoint_url, UAT_CLIENT_ID, UAT_CLIENT_SECRET)
response = get_quote(quotes_endpoint_url, UAT_CLIENT_ID, 10000, access_token, client_dict)

In [48]:
response.json()

{'action': 'INITIAL_SALES',
 'availablePlanPrices': [{'planCode': 'PL-SMED-21',
   'genericName': 'Visitors to Canada Medical',
   'productCode': 'PR-VIS-1',
   'effectiveDate': '2026-02-01',
   'expiryDate': '2026-03-08',
   'isExtension': False,
   'minimumApplied': False,
   'hasPriceOverridden': False,
   'ratedPrice': {'type': 'DEBIT',
    'currencyCode': 'CAD',
    'total': 87.12,
    'comm': 0.0,
    'commPct': 0.0,
    'net': 0.0},
   'chargedPrice': {'type': 'DEBIT',
    'currencyCode': 'CAD',
    'total': 87.12,
    'comm': 0.0,
    'commPct': 0.0,
    'net': 0.0},
   'priceInputParameters': [{'code': 'PM-VSPT-13',
     'value': None,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-VSPT-12',
     'value': None,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-SPORT-8',
     'value': None,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-SPORT11',
     'value': None,
     'grossPremium': None,
     

In [20]:
'messages' in response.json()

False

In [12]:
# function to print shapes on a slide
def print_shapes(slide):
    index = 0
    for shape in slide.shapes:
        print("\nIndex: ",index)
        print("Has Table: ",shape.has_table)
        print("Has Text Frame: ", shape.has_text_frame)
        if shape.has_text_frame:
            print("Text Value: ",shape.text_frame.text)
        print("Object: ",shape)
        index = index + 1

def set_text(run, text, font_name, size, color='Black', bold=False, italic=False):
    """
    Sets the text of a 'run' with style parameters
    Returns: a run
    """
    run.text = text
    # setting font style
    font = run.font
    font.name = font_name
    font.size = size
    font.color.theme_color = color
    font.bold = bold
    font.italic = italic
    return run

def set_cover(text_frame, text):
    text_frame.clear()
    paragraph = text_frame.paragraphs[0]
    run = paragraph.add_run()
    set_text(run, text, FONT, Pt(11), color=MSO_THEME_COLOR.ACCENT_6)

In [ ]:

### Cover Formatting Info ###
# Font Size: 11
# Font Name: Segoe UI
# Bold: Yes
# Font Color: 172, 110, 223

# get first slide
cover_slide = prs.slides[0]

# setting client name on cover
name_text_frame = cover_slide.shapes[1].text_frame
set_cover(name_text_frame, client_dict["full_name"])

# setting client date of birth on cover
dob_text_frame = cover_slide.shapes[2].text_frame
set_cover(dob_text_frame, client_dict["dob_1"])

# setting client email on cover
email_text_frame = cover_slide.shapes[3].text_frame
set_cover(email_text_frame, client_dict["email"])

# setting client destination on cover
destination_text_frame = cover_slide.shapes[4].text_frame
set_cover(destination_text_frame, client_dict["dest_country"])

for i in range(1,5):
    print(prs.slides[0].shapes[i].text_frame.text)

Dea Test
Nov 29, 1980
andrea.britto@gmail.com
Canadá


In [27]:
def fill_table(table,some_list):
    for x in range(0,len(some_list)):
        # get cell in the second row, first column (i.e., skip table header)
        coverage_textframe = table.cell(x+1,0).text_frame
        # clear whatever is in text frame
        coverage_textframe.clear()
        # by default always have 1 paragraph when empty, grab that paragraph
        paragraph = coverage_textframe.paragraphs[0]
        # add a new run to it
        run = paragraph.add_run()
        # set the text to be added
        coverage_text = f"${some_list[x][0]:,}"
        # setting text value
        set_text(run, coverage_text, 'Segoe UI', Pt(10.5), color=MSO_THEME_COLOR.TEXT_1)
        
        # get cell in second column, the second row (i.e., skip table header)
        quote_textframe = table.cell(x+1,1).text_frame
        # clear whatever is in text frame
        quote_textframe.clear()
        # by default always have 1 paragraph when empty, grab that paragraph
        paragraph = quote_textframe.paragraphs[0]
        # add a new run to it
        run = paragraph.add_run()
        # set the text to be added
        quote_text = "$" + str(some_list[x][1])
        # setting text value
        set_text(run, quote_text, 'Segoe UI', Pt(10.5), color=MSO_THEME_COLOR.TEXT_1)

def print_table(table):
    for row in range(len(table.rows)):
        for col in range(len(table.columns)):
            print(table.cell(row, col).text, end=" ")
        print()

In [5]:
client_dict

{'full_name': 'Dea Test',
 'email': 'andrea.britto@gmail.com',
 'submit': '2026-01-09',
 'trip_start': '2026-02-01',
 'dest_arrival': '2026-02-02',
 'end': '2026-03-08',
 'dest_country': 'Canadá',
 'depart_region': '_Fora do Canadá',
 'num_travellers': 1,
 'name_1': 'Dea Test',
 'dob_1': '1980-11-29',
 'name_2': nan,
 'dob_2': nan,
 'name_3': nan,
 'dob_3': nan,
 'name_4': nan,
 'dob_4': nan,
 'name_5': nan,
 'dob_5': nan}

In [ ]:
access_token = get_access_token()
get_quote(10000, access_token, client_dict)

{'action': 'INITIAL_SALES',
 'availablePlanPrices': [{'planCode': 'PL-SMED-21',
   'genericName': 'Visitors to Canada Medical',
   'productCode': 'PR-VIS-1',
   'effectiveDate': '2026-02-01',
   'expiryDate': '2026-03-08',
   'isExtension': False,
   'minimumApplied': False,
   'hasPriceOverridden': False,
   'ratedPrice': {'type': 'DEBIT',
    'currencyCode': 'CAD',
    'total': 78.84,
    'comm': 0.0,
    'commPct': 0.0,
    'net': 0.0},
   'chargedPrice': {'type': 'DEBIT',
    'currencyCode': 'CAD',
    'total': 78.84,
    'comm': 0.0,
    'commPct': 0.0,
    'net': 0.0},
   'priceInputParameters': [{'code': 'PM-VSPT-19',
     'value': None,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-DED-45',
     'value': 0,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-SPORT-8',
     'value': None,
     'grossPremium': None,
     'overriddenPrice': None},
    {'code': 'PM-VSPT-18',
     'value': None,
     'grossPremium': None,
     'ove

In [ ]:

# tugo slide table filling

# get slide
tugo_slide = prs.slides[2]

# get the table from the slide
tugo_quotes = tugo_slide.shapes[6].table

# get access token to request quotes price
access_token = get_access_token()

# Create list with quotes for each coverage levels
    # inner list represents a row in table
    # value in each inner list is value in each row
coverage_levels = [10000, 25000, 50000, 100000, 200000, 300000, 500000]

quotes_list = []
for coverage in coverage_levels:
    try:
        quote = get_quote(coverage, access_token, client_dict)['availablePlanPrices'][0]['ratedPrice']['total']
        table_row = [coverage, quote]
        quotes_list.append(table_row)
    except Exception as e:
        raise e

# print before
print_table(tugo_quotes)

# filling table
fill_table(tugo_quotes,quotes_list)

# print after filling
print_table(tugo_quotes)


COBERTURA
(dólares canadenses) PREÇO
($CAD - dólares canadenses) 
$10,000 $None 
$25,000 $None 
$50,000 $None 
$100,000 $None 
$200,000 $None 
$300,000 $None 
$500,000 $None 
Accidental Death and Dismemberment  (opcional) + $ 
COBERTURA
(dólares canadenses) PREÇO
($CAD - dólares canadenses) 
$10,000 $78.84 
$25,000 $114.48 
$50,000 $130.32 
$100,000 $193.32 
$200,000 $269.28 
$300,000 $344.52 
$500,000 $379.08 
Accidental Death and Dismemberment  (opcional) + $ 


In [ ]:
# Setting info on Tugo slide

tugo_info_header = tugo_slide.shapes[25].table

# setting seguro dates
tugo_dates_text_frame = tugo_info_header.cell(0,1).text_frame
tugo_dates_text_frame.clear()
d_para = tugo_dates_text_frame.paragraphs[0]
d_run = d_para.add_run()
d_text = client_dict["start"] + " - " + client_dict["end"]
set_text(d_run, d_text, FONT, Pt(9), MSO_THEME_COLOR.ACCENT_2)

# setting number of segurados
tugo_insured_text_frame = tugo_info_header.cell(0,3).text_frame
tugo_insured_text_frame.clear()
d_para = tugo_insured_text_frame.paragraphs[0]
d_run = d_para.add_run()
num_travelers = client_dict["num_travellers"]
d_text = str(num_travelers) + " pessoa" + 's'*(num_travelers-1)
set_text(d_run, d_text, FONT, Pt(9), color=MSO_THEME_COLOR.ACCENT_2)

print_table(tugo_info_header)

Data do seguro: Sep 25, 2025 - Oct 4, 2025 Preços totais para:  1 pessoa 


In [20]:
prs.save('template.pptx')